In [1]:
import pandas as pd
import numpy as np

accounts = pd.read_csv("vyapar_accounts.csv", parse_dates=["signup_date", "upgrade_date"])
usage = pd.read_csv("vyapar_monthly_usage.csv", parse_dates=["month"])

print(accounts.shape, usage.shape)
print(accounts.dtypes)
print(usage.dtypes)
print("\nDuplicate rows in usage:", usage.duplicated().sum())
print("Negative usage values:", (usage["eway_bills_used"] < 0).sum())
print("City tier values:", accounts["city_tier"].unique())

(500, 9) (2969, 5)
account_id                          object
business_name                       object
vertical                            object
city_tier                           object
signup_date                 datetime64[ns]
firms_count                          int64
support_tickets_last_6mo             int64
upgrade_date                datetime64[ns]
plan_tier                           object
dtype: object
account_id                        object
month                     datetime64[ns]
eway_bills_used                    int64
firms_count_that_month             int64
hit_eway_cap                        bool
dtype: object

Duplicate rows in usage: 29
Negative usage values: 3
City tier values: ['Tier 3' 'Tier 2' 'Tier 1' 'tier 1' 'tier 2' 'tier 3']


In [2]:
#drop duplicates
before = len(usage)
usage = usage.drop_duplicates()
print(f"Dropped {before - len(usage)} duplicate rows")

Dropped 29 duplicate rows


In [3]:
usage.loc[usage["eway_bills_used"] < 0, "eway_bills_used"] = np.nan

usage = usage.sort_values(["account_id", "month"])
usage["eway_bills_used"] = (
    usage.groupby("account_id")["eway_bills_used"]
    .transform(lambda s: s.interpolate(limit_direction="both"))
)

print("Remaining nulls after interpolation:", usage["eway_bills_used"].isna().sum())
usage["eway_bills_used"] = usage["eway_bills_used"].round().astype(int)

Remaining nulls after interpolation: 0


In [4]:
accounts["city_tier"] = accounts["city_tier"].str.strip().str.title()
print(accounts["city_tier"].unique())

['Tier 3' 'Tier 2' 'Tier 1']


In [5]:
all_months = pd.date_range(usage["month"].min(), usage["month"].max(), freq="MS")
full_grid = pd.MultiIndex.from_product(
    [accounts["account_id"], all_months], names=["account_id", "month"]
).to_frame(index=False)

usage_full = full_grid.merge(usage, on=["account_id", "month"], how="left")
usage_full["was_missing"] = usage_full["eway_bills_used"].isna()

print("Missing account-months:", usage_full["was_missing"].sum())

# Impute missing months the same way — interpolate within each account's own series
usage_full["eway_bills_used"] = (
    usage_full.groupby("account_id")["eway_bills_used"]
    .transform(lambda s: s.interpolate(limit_direction="both"))
)
usage_full["firms_count_that_month"] = (
    usage_full.groupby("account_id")["firms_count_that_month"]
    .transform(lambda s: s.ffill().bfill())
)
usage_full["hit_eway_cap"] = usage_full["eway_bills_used"] >= 10

Missing account-months: 60


In [6]:
accounts["is_upgrader"] = accounts["plan_tier"].eq("Gold")
accounts["days_to_upgrade"] = (accounts["upgrade_date"] - accounts["signup_date"]).dt.days
accounts["tenure_days"] = (usage["month"].max() - accounts["signup_date"]).dt.days

usage_full["pct_of_eway_cap"] = usage_full["eway_bills_used"] / 10

In [7]:
accounts.head()

,account_id,business_name,vertical,city_tier,signup_date,firms_count,support_tickets_last_6mo,upgrade_date,plan_tier,is_upgrader,days_to_upgrade,tenure_days
0,VYP100000,"Choudhury, Bakshi and Maharaj",IT Services,Tier 3,2024-06-11,2,2,2026-06-01,Gold,True,720.0,812
1,VYP100001,Kapoor and Sons,Construction,Tier 3,2026-01-20,1,2,2026-08-01,Gold,True,193.0,224
2,VYP100002,"Chaudry, Chahal and Sami",Garments,Tier 3,2025-05-23,3,2,2026-08-01,Gold,True,435.0,466
3,VYP100003,Balan Inc,Tours & Travel,Tier 2,2025-04-10,2,1,NaT,Silver,False,NaN,509
4,VYP100004,Kaul PLC,Grocery,Tier 2,2024-08-08,1,2,2026-06-01,Gold,True,662.0,754


In [8]:
issues = {}

issues["duplicate_account_ids"] = accounts["account_id"].duplicated().sum()

tier_mismatch = accounts[
    (accounts["plan_tier"] == "Gold") & (accounts["upgrade_date"].isna())
    | (accounts["plan_tier"] == "Silver") & (accounts["upgrade_date"].notna())
]
issues["plan_tier_vs_upgrade_date_mismatches"] = len(tier_mismatch)

bad_dates = accounts[accounts["upgrade_date"] < accounts["signup_date"]]
issues["upgrade_before_signup"] = len(bad_dates)

gold_accounts = accounts.loc[accounts["is_upgrader"], "account_id"]
gold_usage = usage_full[usage_full["account_id"].isin(gold_accounts)]
issues["gold_accounts_over_5_firms"] = (gold_usage["firms_count_that_month"] > 5).sum()

silver_accounts = accounts.loc[~accounts["is_upgrader"], "account_id"]
silver_usage = usage_full[usage_full["account_id"].isin(silver_accounts)]
issues["silver_accounts_over_3_firms"] = (silver_usage["firms_count_that_month"] > 3).sum()

issues["still_null_after_interpolation"] = usage_full["eway_bills_used"].isna().sum()

for check, count in issues.items():
    flag = "⚠️ CHECK THIS" if count > 0 else "OK"
    print(f"{check}: {count}  [{flag}]")

duplicate_account_ids: 0  [OK]
plan_tier_vs_upgrade_date_mismatches: 0  [OK]
upgrade_before_signup: 0  [OK]
gold_accounts_over_5_firms: 0  [OK]
silver_accounts_over_3_firms: 0  [OK]
still_null_after_interpolation: 0  [OK]


In [ ]:
from sqlalchemy import create_engine
# Step 1: Connect to PostgreSQL
# Replace placeholders with your actual details

username = "postgres"        # default user
password = "Ss16132484"         # the password you set during installation
host = "localhost"           # if running locally
port = "5432"                # default PostgreSQL port
database = "vyapar_db"  # the database you created in pgAdmin

from sqlalchemy import create_engine

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# Step 2: Load DataFrame into PostgreSQL
table_name = "customer"   # choose any table name

df.to_sql(table_name, engine, if_exists="replace", index=False)
 
print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")

In [9]:
pip install psycopg2-binary sqlalchemy 



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
!pip install sqlalchemy psycopg2-binary


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from getpass import getpass
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text

username = "postgres"
password = quote_plus(getpass("Postgres password: "))
host = "localhost"
port = "5432"
database = "vyapar_db"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# Test: this is where a wrong password or database name will show up
with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database();")).scalar())

Postgres password:  ········


vyapar_db


In [12]:
accounts.to_sql("accounts", engine, if_exists="replace", index=False)
usage_full.to_sql("monthly_usage", engine, if_exists="replace", index=False)

print("Loaded: accounts, monthly_usage")

Loaded: accounts, monthly_usage


In [13]:
with engine.connect() as conn:
    print("accounts rows:", conn.execute(text("SELECT COUNT(*) FROM accounts")).scalar())
    print("monthly_usage rows:", conn.execute(text("SELECT COUNT(*) FROM monthly_usage")).scalar())

accounts rows: 500
monthly_usage rows: 3000
